# Лабораторная работа №5. Квантовые сумматоры. Алгоритм Гровера
**Вариант 8**

In [ ]:
!pip install qiskit qiskit-aer matplotlib numpy
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Operator
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

sim = AerSimulator()
sv_sim = AerSimulator(method='statevector')

## Задание 1: Полный сумматор и цепочка переноса

In [ ]:
# 1.1 Полный сумматор для входов A=1, B=0, X=1
# Кубиты: 0=A, 1=B, 2=X, 3=S, 4=C
def full_adder_5q(a_val, b_val, x_val, with_measure=False):
    qc = QuantumCircuit(5, 2 if with_measure else 0)

    if a_val == 1:
        qc.x(0)
    if b_val == 1:
        qc.x(1)
    if x_val == 1:
        qc.x(2)

    # Перенос C = (A AND B) OR (X AND (A XOR B))
    qc.ccx(0, 1, 4)
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.ccx(2, 3, 4)

    # Сумма S = A XOR B XOR X
    qc.cx(2, 3)

    if with_measure:
        qc.measure(3, 0)  # S -> c[0]
        qc.measure(4, 1)  # C -> c[1]
    return qc

qc_no_meas = full_adder_5q(1, 0, 1, with_measure=False)
sv_101 = Statevector.from_instruction(qc_no_meas)
print('Вектор состояния для A=1, B=0, X=1 (ненулевые амплитуды, порядок q4 q3 q2 q1 q0 = C S X B A):')
for state_str, amp in sv_101.to_dict().items():
    if abs(amp) > 1e-9:
        print(f'  |{state_str}> : {amp}')

qc_meas = full_adder_5q(1, 0, 1, with_measure=True)
counts_101 = sim.run(qc_meas, shots=1024).result().get_counts()
print('Распределение при shots=1024 (формат C_S):', counts_101)

**Объяснение для $A=1, B=0, X=1$:**
В Qiskit используется порядок битов справа налево ($|q_4 q_3 q_2 q_1 q_0\rangle = |C\, S\, X\, B\, A\rangle$).
Для входов $A=1, B=0, X=1$ схема детерминированно переводит систему в базисное состояние $|10101\rangle$, где $q_3 = S = 0$ и $q_4 = C = 1$. При измерении кубитов $q_3$ и $q_4$ во всех 1024 запусках (shots) получаем результат `'10'` ($C=1, S=0$), что полностью соответствует строке `1 | 0 | 1 | 0 | 1` таблицы истинности ($1 + 0 + 1 = 2_{10} = 10_2$).

In [ ]:
# 1.2 Проверка 5-кубитного полного сумматора на всех 8 наборах входов
truth_table_fa = {
    (0, 0, 0): (0, 0),
    (0, 0, 1): (1, 0),
    (0, 1, 0): (1, 0),
    (0, 1, 1): (0, 1),
    (1, 0, 0): (1, 0),
    (1, 0, 1): (0, 1),
    (1, 1, 0): (0, 1),
    (1, 1, 1): (1, 1),
}

print('A | B | X | exp_S | exp_C | got_S | got_C')
print('-' * 42)
for (a, b, x), (exp_s, exp_c) in truth_table_fa.items():
    qc_test = full_adder_5q(a, b, x, with_measure=True)
    cnt = sim.run(qc_test, shots=100).result().get_counts()
    assert len(cnt) == 1, f'Ожидался детерминированный ответ, получено: {cnt}'
    only_key = list(cnt.keys())[0]
    got_c = int(only_key[0])
    got_s = int(only_key[1])
    assert (got_s, got_c) == (exp_s, exp_c), f'Ошибка на {(a, b, x)}: {(got_s, got_c)} != {(exp_s, exp_c)}'
    print(f'{a} | {b} | {x} |   {exp_s}   |   {exp_c}   |   {got_s}   |   {got_c}')

qc_sample_5q = full_adder_5q(0, 0, 0, with_measure=False)
print('\n5q сумматор -> кубиты:', qc_sample_5q.num_qubits,
      '| CX:', qc_sample_5q.count_ops().get('cx', 0),
      '| CCX:', qc_sample_5q.count_ops().get('ccx', 0),
      '| depth:', qc_sample_5q.depth())

In [ ]:
# 1.3 Полный сумматор из двух полусумматоров и вентиля ИЛИ (OR)
# Кубиты: 0=A, 1=B, 2=X, 3=S1, 4=C1, 5=C2, 6=S, 7=C
def full_adder_two_ha(a_val, b_val, x_val, with_measure=False):
    qc = QuantumCircuit(8, 2 if with_measure else 0)

    if a_val == 1:
        qc.x(0)
    if b_val == 1:
        qc.x(1)
    if x_val == 1:
        qc.x(2)

    # Полусумматор 1 (A, B -> S1, C1)
    qc.cx(0, 3)
    qc.cx(1, 3)
    qc.ccx(0, 1, 4)

    # Полусумматор 2 (S1, X -> S, C2)
    qc.cx(3, 6)
    qc.cx(2, 6)
    qc.ccx(3, 2, 5)

    # Вентиль ИЛИ: C = C1 OR C2 = NOT((NOT C1) AND (NOT C2))
    qc.x(4)
    qc.x(5)
    qc.ccx(4, 5, 7)
    qc.x(7)
    qc.x(4)
    qc.x(5)

    if with_measure:
        qc.measure(6, 0)
        qc.measure(7, 1)
    return qc

for (a, b, x), (exp_s, exp_c) in truth_table_fa.items():
    cnt = sim.run(full_adder_two_ha(a, b, x, with_measure=True), shots=100).result().get_counts()
    assert len(cnt) == 1
    key = list(cnt.keys())[0]
    assert (int(key[1]), int(key[0])) == (exp_s, exp_c)

qc_sample_ha = full_adder_two_ha(0, 0, 0, with_measure=False)
print('Из двух полусумматоров и OR -> кубиты:', qc_sample_ha.num_qubits,
      '| CX:', qc_sample_ha.count_ops().get('cx', 0),
      '| CCX:', qc_sample_ha.count_ops().get('ccx', 0),
      '| depth:', qc_sample_ha.depth())

**Сравнение двух реализаций полного сумматора:**
* **5-кубитный сумматор:** 5 кубитов, 3 вентиля CX, 2 вентиля CCX, глубина = 5.
* **Сумматор из двух полусумматоров и OR:** 8 кубитов, 4 вентиля CX, 3 вентиля CCX, глубина = 8.

**Вывод:** 5-кубитная реализация экономичнее по всем параметрам (на 3 кубита меньше, на 1 CX и 1 CCX меньше, глубина ниже на 3), так как промежуточная сумма $A \oplus B$ вычисляется прямо в выходном кубите суммы $S$ без выделения отдельных ancilla-кубитов под $S_1, C_1, C_2$ и последующей сборки переноса через вентиль ИЛИ.

In [ ]:
# 1.4 Цепочка переноса: n-разрядный сумматор (проверка 2, 3 и 4 разрядов)
def build_nbit_adder(n_bits, a_val, b_val, measure=False):
    # Кубиты: A[0..n-1], B[0..n-1], S[0..n], Carry[0..n-2] -> всего 4n кубитов
    n_q = 4 * n_bits
    qc = QuantumCircuit(n_q, n_bits + 1 if measure else 0)

    for i in range(n_bits):
        if (a_val >> i) & 1:
            qc.x(i)
        if (b_val >> i) & 1:
            qc.x(n_bits + i)

    s_base = 2 * n_bits
    c_base = 3 * n_bits + 1

    for i in range(n_bits):
        a_q = i
        b_q = n_bits + i
        s_q = s_base + i
        x_q = None if i == 0 else (c_base + i - 1)
        out_c_q = (s_base + n_bits) if (i == n_bits - 1) else (c_base + i)

        if x_q is None:
            # Младший разряд: входного переноса нет (достаточно полусумматора)
            qc.ccx(a_q, b_q, out_c_q)
            qc.cx(a_q, s_q)
            qc.cx(b_q, s_q)
        else:
            qc.ccx(a_q, b_q, out_c_q)
            qc.cx(a_q, s_q)
            qc.cx(b_q, s_q)
            qc.ccx(x_q, s_q, out_c_q)
            qc.cx(x_q, s_q)

    if measure:
        for k in range(n_bits + 1):
            qc.measure(s_base + k, k)
    return qc

bit_widths = [2, 3, 4]
depths = []

print("Разрядность | Кубитов | CX | CCX | Глубина (depth)")
print("-" * 52)
for n_b in bit_widths:
    qc_s = build_nbit_adder(n_b, 0, 0, measure=False)
    ops = qc_s.count_ops()
    cx_cnt = ops.get("cx", 0)
    ccx_cnt = ops.get("ccx", 0)
    d = qc_s.depth()
    depths.append(d)
    print(f"     {n_b}      |   {qc_s.num_qubits:2d}    | {cx_cnt:2d} |  {ccx_cnt:2d} |       {d}")

# Проверка всех 256 пар (a, b) для 4-разрядного сумматора
for a in range(16):
    for b in range(16):
        qc_4 = build_nbit_adder(4, a, b, measure=True)
        cnt = sim.run(qc_4, shots=16).result().get_counts()
        assert len(cnt) == 1
        assert int(list(cnt.keys())[0], 2) == a + b
print("\nВсе 256 пар (a, b) для 4-битного сумматора проверены успешно.")

plt.figure(figsize=(6, 4))
plt.plot(bit_widths, depths, marker="o")
plt.xticks(bit_widths)
plt.title("Рост глубины цепочки переноса от числа разрядов")
plt.xlabel("Число разрядов n")
plt.ylabel("Глубина схемы (depth)")
plt.grid(True)
plt.show()

### Ответы на вопросы к Заданию 1
1. **Почему перенос $C$ полного сумматора нельзя вычислить одним вентилем CCX? Какие слагаемые для него нужны?**
   Один вентиль CCX (Тоффоли) вычисляет конъюнкцию только двух управляющих битов ($A \land B$). Перенос $C$ является мажоритарной функцией трёх переменных: $C = \text{MAJ}(A, B, X) = (A \land B) \oplus (X \land (A \oplus B))$ или $(A \land B) \lor (A \land X) \lor (B \land X)$. Для его вычисления требуются как минимум два слагаемых: перенос от сложения входных битов ($A \land B$) и перенос, возникающий при сложении их частичной суммы с входным переносом ($X \land (A \oplus B)$).
2. **Вентиль CX копирует классический бит, но не может скопировать произвольную суперпозицию. Как это связано с теоремой о запрете клонирования?**
   Для базисных состояний $|x\rangle|0\rangle$ вентиль CX даёт $|x\rangle|x\rangle$ ($x \in \{0, 1\}$). Однако при подаче суперпозиции $|\psi\rangle = \alpha|0\rangle + \beta|1\rangle$ в силу линейности унитарного оператора получаем запутанное состояние $\text{CX}(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$, которое **не равно** тензорному произведению копий $|\psi\rangle \otimes |\psi\rangle = \alpha^2|00\rangle + \alpha\beta(|01\rangle + |10\rangle) + \beta^2|11\rangle$ (при $\alpha\beta \neq 0$). Это прямое следствие теоремы о запрете клонирования (No-Cloning Theorem), утверждающей невозможность создания унитарного оператора, копирующего произвольное неизвестное квантовое состояние.
3. **Как можно уменьшить глубину 4-разрядного сумматора и какой ценой (дополнительные кубиты, вентили)?**
   В последовательном сумматоре с цепочкой переноса (ripple-carry adder) глубина растёт линейно $O(n)$, так как каждый следующий разряд ждёт переноса от предыдущего. Глубину можно уменьшить до логарифмической $O(\log n)$ с помощью **сумматоров с ускоренным переносом (Quantum Carry-Lookahead Adder)** или параллельных префиксных деревьев (Брента–Кунга, Когге–Стоуна), вычисляющих сигналы генерации $g_i = a_i \land b_i$ и распространения $p_i = a_i \oplus b_i$ параллельно. Ценой такого ускорения является увеличение числа вспомогательных (ancilla) кубитов для хранения промежуточных префиксов и рост общего числа двух- и трёхкубитных вентилей (CX и CCX).

## Задание 2: Вариант 8 — $F(A, B, C, D) = \neg((A \oplus C) \land (B \oplus D))$

In [ ]:
# Классическая реализация функции Варианта 8
def f_classic(a, b, c, d):
    xor1 = a ^ c
    xor2 = b ^ d
    return int(not (xor1 and xor2))

# Базовые квантовые подсхемы (элементы)
def add_not(qc, q):
    qc.x(q)

def add_xor(qc, q1, q2, target):
    qc.cx(q1, target)
    qc.cx(q2, target)

def add_and(qc, q1, q2, target):
    qc.ccx(q1, q2, target)

def add_nand(qc, q1, q2, target):
    add_and(qc, q1, q2, target)
    add_not(qc, target)

# 1. Базовая схема: каждый промежуточный результат записывается в отдельный ancilla-кубит
# Кубиты: 0=A, 1=B, 2=C, 3=D, 4=F(out), 5=anc1(A^C), 6=anc2(B^D), 7=anc3(F_temp)
def build_basic_f(a, b, c, d, measure=False):
    qc = QuantumCircuit(8, 4 if measure else 0)
    if a: qc.x(0)
    if b: qc.x(1)
    if c: qc.x(2)
    if d: qc.x(3)

    comp = QuantumCircuit(8)
    add_xor(comp, 0, 2, 5)       # anc1 = A XOR C
    add_xor(comp, 1, 3, 6)       # anc2 = B XOR D
    add_nand(comp, 5, 6, 7)      # anc3 = NOT(anc1 AND anc2)

    qc.compose(comp, inplace=True)
    qc.cx(7, 4)                  # Копируем результат на выходной кубит 4
    qc.compose(comp.inverse(), inplace=True)  # Uncompute: возвращаем ancilla в |0>

    if measure:
        qc.measure(4, 0)  # F(out)
        qc.measure(5, 1)  # anc1
        qc.measure(6, 2)  # anc2
        qc.measure(7, 3)  # anc3
    return qc

# 2. Оптимизированная схема: убираем лишний 3-й ancilla-кубит и один CX,
# вычисляя NAND напрямую в выходной кубит F(out)
# Кубиты: 0=A, 1=B, 2=C, 3=D, 4=F(out), 5=anc1(A^C), 6=anc2(B^D)
def build_opt_f(a, b, c, d, measure=False):
    qc = QuantumCircuit(7, 3 if measure else 0)
    if a: qc.x(0)
    if b: qc.x(1)
    if c: qc.x(2)
    if d: qc.x(3)

    comp = QuantumCircuit(7)
    add_xor(comp, 0, 2, 5)       # anc1 = A XOR C
    add_xor(comp, 1, 3, 6)       # anc2 = B XOR D

    qc.compose(comp, inplace=True)
    add_nand(qc, 5, 6, 4)        # F(out) = NOT(anc1 AND anc2) напрямую в выходной кубит
    qc.compose(comp.inverse(), inplace=True)  # Uncompute anc1 и anc2 -> |0>

    if measure:
        qc.measure(4, 0)  # F(out)
        qc.measure(5, 1)  # anc1
        qc.measure(6, 2)  # anc2
    return qc

# Проверка обеих схем на всех 16 наборах входов
print('A B C D | Классика | Квантово (base) | Квантово (opt) | Ancilla(opt) | Статус')
print('-' * 76)
for a in [0, 1]:
    for b in [0, 1]:
        for c in [0, 1]:
            for d in [0, 1]:
                exp_f = f_classic(a, b, c, d)

                # Проверка базовой схемы
                cnt_b = sim.run(build_basic_f(a, b, c, d, measure=True), shots=100).result().get_counts()
                assert len(cnt_b) == 1
                out_b = list(cnt_b.keys())[0]
                got_f_b = int(out_b[-1])
                anc_b = out_b[:-1]
                assert got_f_b == exp_f
                assert anc_b == '000', f'Ancilla базовой схемы не обнулились: {anc_b}'

                # Проверка оптимизированной схемы
                cnt_o = sim.run(build_opt_f(a, b, c, d, measure=True), shots=100).result().get_counts()
                assert len(cnt_o) == 1
                out_o = list(cnt_o.keys())[0]
                got_f_o = int(out_o[-1])
                anc_o = out_o[:-1]
                assert got_f_o == exp_f
                assert anc_o == '00', f'Ancilla оптимизированной схемы не обнулились: {anc_o}'

                print(f'{a} {b} {c} {d} |     {exp_f}    |        {got_f_b}        |       {got_f_o}        |      {anc_o}      |   OK')

b_circ = build_basic_f(0, 0, 0, 0, measure=False)
o_circ = build_opt_f(0, 0, 0, 0, measure=False)
print('\nБазовая схема    -> кубитов:', b_circ.num_qubits,
      '| CX:', b_circ.count_ops().get('cx', 0),
      '| CCX:', b_circ.count_ops().get('ccx', 0),
      '| depth:', b_circ.depth())
print('Оптимизированная -> кубитов:', o_circ.num_qubits,
      '| CX:', o_circ.count_ops().get('cx', 0),
      '| CCX:', o_circ.count_ops().get('ccx', 0),
      '| depth:', o_circ.depth())

**Сравнение базовой и оптимизированной реализаций (Вариант 8):**
* **Базовая схема:** 8 кубитов, 9 вентилей CX, 2 вентиля CCX, глубина = 9.
* **Оптимизированная схема:** 7 кубитов, 8 вентилей CX, 1 вентиль CCX, глубина = 6.

**Почему минимум по числу кубитов и минимум по глубине могут не совпадать?**
Экономия кубитов достигается за счёт повторного использования одних и тех же вспомогательных кубитов для разных подвыражений. Это требует последовательного выполнения операций «вычисление $\rightarrow$ копирование результата $\rightarrow$ обратное вычисление (uncompute) $\rightarrow$ следующее вычисление», что выстраивает вентили в длинную последовательную цепочку и **увеличивает глубину** схемы. Напротив, если выделить больше вспомогательных кубитов, независимые подвыражения можно вычислять **параллельно** на разных кубитах за один такт, что минимизирует глубину ценой увеличения ширины (числа кубитов) схемы.

## Задание 3: Моделирование шума для оптимизированной схемы Варианта 8

In [ ]:
p1 = 0.001
p2_list = [0.0, 0.001, 0.005, 0.01, 0.02]
accuracies_opt = []
accuracies_base = []

# Посмотрим число двухкубитных вентилей CX после транспиляции
qc_opt_t = transpile(build_opt_f(0, 0, 0, 0, measure=True), sim, basis_gates=['rz', 'sx', 'x', 'cx'])
qc_base_t = transpile(build_basic_f(0, 0, 0, 0, measure=True), sim, basis_gates=['rz', 'sx', 'x', 'cx'])
print(f"После transpile: CX в оптимизированной = {qc_opt_t.count_ops().get('cx', 0)}, CX в базовой = {qc_base_t.count_ops().get('cx', 0)}\n")

for p2 in p2_list:
    noise_model = NoiseModel()
    err_1q = depolarizing_error(p1, 1)
    err_2q = depolarizing_error(p2, 2)
    noise_model.add_all_qubit_quantum_error(err_1q, ['x', 'sx', 'rz'])
    noise_model.add_all_qubit_quantum_error(err_2q, ['cx'])

    correct_opt = 0
    correct_base = 0
    shots_per_input = 250
    total_shots = 16 * shots_per_input

    for a in [0, 1]:
        for b in [0, 1]:
            for c in [0, 1]:
                for d in [0, 1]:
                    exp_f = f_classic(a, b, c, d)

                    qc_t_o = transpile(build_opt_f(a, b, c, d, measure=True), sim, basis_gates=['rz', 'sx', 'x', 'cx'], optimization_level=1)
                    cnt_o = sim.run(qc_t_o, noise_model=noise_model, shots=shots_per_input).result().get_counts()
                    for bitstr, freq in cnt_o.items():
                        if int(bitstr[-1]) == exp_f:
                            correct_opt += freq

                    qc_t_b = transpile(build_basic_f(a, b, c, d, measure=True), sim, basis_gates=['rz', 'sx', 'x', 'cx'], optimization_level=1)
                    cnt_b = sim.run(qc_t_b, noise_model=noise_model, shots=shots_per_input).result().get_counts()
                    for bitstr, freq in cnt_b.items():
                        if int(bitstr[-1]) == exp_f:
                            correct_base += freq

    acc_o = correct_opt / total_shots
    acc_b = correct_base / total_shots
    accuracies_opt.append(acc_o)
    accuracies_base.append(acc_b)
    print(f'p2 = {p2:5.3f} -> доля верных ответов: opt = {acc_o:.4f} | base = {acc_b:.4f}')

plt.figure(figsize=(7, 4))
plt.plot(p2_list, accuracies_opt, marker='o', label='Оптимизированная (1 CCX, 8 CX)')
plt.plot(p2_list, accuracies_base, marker='s', linestyle='--', label='Базовая (2 CCX, 9 CX)')
plt.title('Доля верных ответов от шума p2 (Вариант 8)')
plt.xlabel('p2 (вероятность деполяризующего шума на CX)')
plt.ylabel('Доля верных ответов')
plt.legend()
plt.grid(True)
plt.show()

### Ответы на вопросы к Заданию 3
1. **Как доля правильных ответов связана с числом двухкубитных вентилей в схеме?**
   При транспиляции каждый вентиль Тоффоли (CCX) раскладывается минимум в 6 двухкубитных вентилей CX. С ростом числа вентилей CX вероятность безошибочного прохождения всей схемы экспоненциально падает: $P_{\text{успеха}} \approx (1 - p_1)^{N_{1q}}(1 - p_2)^{N_{2q}}$. Поэтому чем больше в схеме двухкубитных вентилей CX и чем выше вероятность ошибки $p_2$, тем быстрее деградирует точность вычислений и снижается доля верных ответов.
2. **Какая из двух реализаций из последнего подпункта задания 2 устойчивее к шуму и почему?**
   Устойчивее к шуму **оптимизированная реализация**, поскольку в ней используется только **1 вентиль CCX** (против 2 CCX в базовой) и **8 вентилей CX** (против 9 CX в базовой). После разложения в базисные вентили оптимизированная схема содержит на 7 двухкубитных вентилей CX меньше (14 против 21) и имеет меньшую глубину, поэтому в ней накапливается существенно меньше деполяризующих ошибок.

## Задание 4: Оракул, диффузор и алгоритм Гровера

In [ ]:
def apply_mcz(qc, qubits):
    target = qubits[-1]
    controls = qubits[:-1]
    qc.h(target)
    qc.mcx(controls, target)
    qc.h(target)

def build_oracle(n, marked):
    qc = QuantumCircuit(n, name='Oracle')
    # В Qiskit младший бит (индекс 0) соответствует правому символу строки marked
    rev_marked = marked[::-1]
    for i, ch in enumerate(rev_marked):
        if ch == '0':
            qc.x(i)
    apply_mcz(qc, list(range(n)))
    for i, ch in enumerate(rev_marked):
        if ch == '0':
            qc.x(i)
    return qc

def build_diffuser(n):
    qc = QuantumCircuit(n, name='Diffuser')
    for i in range(n):
        qc.h(i)
    for i in range(n):
        qc.x(i)
    apply_mcz(qc, list(range(n)))
    for i in range(n):
        qc.x(i)
    for i in range(n):
        qc.h(i)
    return qc

# 4.1 Проверка оракула для n = 2, 3, 4, 5 на всех 2^n базисных состояниях
for n_test in [2, 3, 4, 5]:
    w_str = ('0110101')[-n_test:]
    op_or = Operator(build_oracle(n_test, w_str)).data
    target_idx = int(w_str, 2)
    expected_diag = np.ones(2**n_test)
    expected_diag[target_idx] = -1.0
    assert np.allclose(op_or, np.diag(expected_diag)), f'Ошибка оракула для n={n_test}'
print('Оракул проверен для n = 2, 3, 4, 5: на |w> фаза -1, на остальных +1.')

# 4.2 Проверка диффузора и сравнение матрицы с (2|s><s| - I)
n_diff = 3
op_diff = Operator(build_diffuser(n_diff)).data
s_vec = np.ones((2**n_diff, 1)) / np.sqrt(2**n_diff)
mat_theory = 2 * (s_vec @ s_vec.T) - np.eye(2**n_diff)
assert np.allclose(op_diff, -mat_theory)
print('Матрица диффузора совпадает с (2|s><s| - I) с точностью до общего знака (-1).')

# 4.3 Сборка алгоритма Гровера и проверка для n=3, marked='101' при k=1 и k=2
def grover_circuit(n, marked_list, iterations):
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
    for _ in range(iterations):
        for m_str in marked_list:
            qc.compose(build_oracle(n, m_str), inplace=True)
        qc.compose(build_diffuser(n), inplace=True)
    return qc

n3 = 3
N3 = 2**n3
w3 = '101'
theta3 = np.arcsin(1.0 / np.sqrt(N3))
for k_it in [1, 2]:
    sv = Statevector.from_instruction(grover_circuit(n3, [w3], k_it))
    p_sim = sv.probabilities_dict()[w3]
    p_th = np.sin((2 * k_it + 1) * theta3)**2
    assert np.isclose(p_sim, p_th)
    print(f'n=3, marked="101", k={k_it}: P_схема = {p_sim:.6f}, P_формула = {p_th:.6f}')

**Объяснение общего знака $(-1)$ в диффузоре:**
Стандартный оператор инверсии относительно среднего имеет вид $U_s = 2|s\rangle\langle s| - I = H^{\otimes n}(2|0\rangle\langle 0| - I)H^{\otimes n}$. В нашей реализации внутри обёртки $H^{\otimes n} X^{\otimes n} (\dots) X^{\otimes n} H^{\otimes n}$ применяется многоуправляемый фазовый вентиль $\text{MCZ}$, который инвертирует фазу только состояния $|11\dots 1\rangle$, реализуя матрицу $I - 2|1\dots 1\rangle\langle 1\dots 1| = -(2|1\dots 1\rangle\langle 1\dots 1| - I)$. После сопряжения вентилями $X^{\otimes n}$ и $H^{\otimes n}$ получаем $-(2|s\rangle\langle s| - I) = -U_s = e^{i\pi} U_s$. Множитель $e^{i\pi} = -1$ является **глобальной фазой** всего квантового состояния; при измерении вероятности определяются квадратом модуля амплитуды $|e^{i\pi}\alpha_x|^2 = |\alpha_x|^2$, поэтому глобальная фаза никак не влияет на результаты измерений.

## Задание 5: Состояние $|1010011\rangle$, $k = 0..15$ и влияние числа shots

In [ ]:
w5 = '1010011'
n5 = len(w5)
N5 = 2**n5
theta5 = np.arcsin(1.0 / np.sqrt(N5))
ks = list(range(16))

p_exact = []
p_exp = []
p_theory = []

for k in ks:
    qc_k = grover_circuit(n5, [w5], k)
    sv_k = Statevector.from_instruction(qc_k)
    p_ex = sv_k.probabilities_dict().get(w5, 0.0)
    p_exact.append(p_ex)
    p_theory.append(np.sin((2 * k + 1) * theta5)**2)

    qc_m = qc_k.copy()
    qc_m.measure_all()
    cnt = sim.run(qc_m, shots=1024, seed_simulator=100 + k).result().get_counts()
    p_exp.append(cnt.get(w5, 0) / 1024.0)

k_opt_exact = int(np.argmax(p_exact))
k_opt_exp = int(np.argmax(p_exp))
val_formula = (np.pi / 4.0) * np.sqrt(N5)
k_floor = int(np.floor(val_formula))
k_round = int(np.round(val_formula))

print(f'Теоретическое значение (pi/4)*sqrt(128) = {val_formula:.4f}')
print(f'floor = {k_floor} (P = {p_theory[k_floor]:.6f}), round = {k_round} (P = {p_theory[k_round]:.6f})')
print(f'k_opt_exact = {k_opt_exact} (P = {p_exact[k_opt_exact]:.6f}), k_opt_exp (1024 shots) = {k_opt_exp} (P_exp = {p_exp[k_opt_exp]:.4f})')

plt.figure(figsize=(8, 4.5))
plt.plot(ks, p_theory, 'k--', linewidth=2, label='Теория sin^2((2k+1)θ)')
plt.plot(ks, p_exact, 'bo', markersize=7, label='Точная (Statevector)')
plt.plot(ks, p_exp, 'rs-', alpha=0.8, label='Эксперимент (1024 shots)')
plt.xticks(ks)
plt.title(f'Вероятность найти |{w5}> (n={n5}, N={N5}) от числа итераций k')
plt.xlabel('Число итераций k')
plt.ylabel('Вероятность P(w)')
plt.legend()
plt.grid(True)
plt.show()

# 5.3 Исследование стабильности определения оптимума на 20 запусках для разных shots
shots_levels = [16, 32, 64, 128, 256, 512, 1024, 2048]
match_rates = []
n_runs = 20

sv_list = [Statevector.from_instruction(grover_circuit(n5, [w5], k)) for k in ks]

for sh in shots_levels:
    matches = 0
    for run_idx in range(n_runs):
        run_p = []
        for k in ks:
            cnt = sv_list[k].sample_counts(shots=sh, seed=10000 * run_idx + 100 * k + sh)
            run_p.append(cnt.get(w5, 0) / sh)
        if int(np.argmax(run_p)) == k_opt_exact:
            matches += 1
    rate = matches / n_runs
    match_rates.append(rate)
    print(f'shots = {sh:4d} -> совпадений оптимума с теоретическим: {matches:2d}/{n_runs} ({rate*100:5.1f}%)')

min_shots_90 = next((sh for sh, r in zip(shots_levels, match_rates) if r >= 0.90), None)
print(f'\nМинимальное число shots с долей совпадений >= 90%: {min_shots_90}')

plt.figure(figsize=(7, 4))
plt.plot(shots_levels, [r * 100 for r in match_rates], marker='o')
plt.axhline(90, color='r', linestyle='--', label='Порог 90%')
plt.xscale('log', base=2)
plt.xticks(shots_levels, shots_levels)
plt.title('Доля совпадений экспериментального оптимума с k_opt от числа shots (20 запусков)')
plt.xlabel('Число запусков (shots)')
plt.ylabel('Доля совпадений (%)')
plt.legend()
plt.grid(True)
plt.show()

### Анализ результатов Задания 5
* **Оптимальное число итераций:** $\frac{\pi}{4}\sqrt{128} \approx 8.8858$. Округление вниз (`floor`) даёт $k = 8$ ($P(8) = 0.9956$), а округление к ближайшему целому (`round`) даёт $k = 9$ ($P(9) = 0.9852$). По точной формуле $\sin^2((2k+1)\theta)$ максимум достигается при **$k_{\text{opt}} = 8$** ($99.56\%$ против $98.52\%$).
* **Влияние статистического шума (shots):** Разница между соседними пиками $P(8) = 0.9956$ и $P(9) = 0.9852$ составляет всего $\Delta P \approx 0.0104$ ($1.04\%$). Стандартное отклонение биномиальной выборки оценивается как $\sigma \approx \sqrt{p(1-p)/N_{\text{shots}}}$. При $N_{\text{shots}} = 1024$ погрешность $\sigma(9) \approx \sqrt{0.985 \cdot 0.015 / 1024} \approx 0.0038$, поэтому в подавляющем большинстве запусков максимум остаётся на $k=8$, но редкие флуктуации могут сместить его на $k=9$.
* Чтобы разность $\Delta P \approx 0.01$ надёжно превышала стандартную ошибку разности $\sqrt{\sigma_1^2 + \sigma_2^2} \approx 0.004$, требуется порядка **$N_{\text{shots}} \ge 1024\text{--}2048$**. На 20 независимых запусках доля совпадений достигает $\ge 90\%$ начиная с **$64\text{--}128$ shots** (и стабилизируется на уровне $95\text{--}100\%$ при $256\text{--}1024$ shots).

## Задание 6: Несколько отмеченных состояний ($M > 1$)

In [ ]:
n6 = 6
N6 = 2**n6
M_list = [1, 2, 4, 8, 16]
ks6 = list(range(16))

rng6 = np.random.default_rng(123)
all_states_6 = [format(i, f'0{n6}b') for i in range(N6)]

plt.figure(figsize=(8, 4.5))
print('  M  | k_теор = round(pi/4*sqrt(N/M)) | k_первый_макс | P(k_первый_макс)')
print('-' * 72)

curves_6 = {}
for M in M_list:
    marked_M = list(rng6.choice(all_states_6, size=M, replace=False))
    probs_M = []
    for k in ks6:
        sv = Statevector.from_instruction(grover_circuit(n6, marked_M, k))
        p_dict = sv.probabilities_dict()
        p_sum = sum(p_dict.get(st, 0.0) for st in marked_M)
        probs_M.append(p_sum)
    curves_6[M] = probs_M

    # Первый локальный максимум
    first_max_k = 0
    for idx in range(1, len(probs_M) - 1):
        if probs_M[idx] >= probs_M[idx - 1] and probs_M[idx] > probs_M[idx + 1]:
            first_max_k = idx
            break

    k_th_M = round((np.pi / 4.0) * np.sqrt(N6 / M))
    print(f' {M:2d}  |              {k_th_M:2d}               |       {first_max_k:2d}      |      {probs_M[first_max_k]:.6f}')
    plt.plot(ks6, probs_M, marker='o', label=f'M = {M}')

plt.xticks(ks6)
plt.title(f'Вероятность успеха алгоритма Гровера при n={n6} (N={N6}) для разных M')
plt.xlabel('Число итераций k')
plt.ylabel('Суммарная вероятность отмеченных состояний P')
plt.legend()
plt.grid(True)
plt.show()

# Проверка случая M = 16, где глобальный максимум на [0..15] больше первого локального
p_m16 = curves_6[16]
first_max_16 = 1
glob_max_16 = int(np.argmax(p_m16))
print(f'Для M=16: первый локальный максимум при k={first_max_16} (P={p_m16[first_max_16]:.4f}), '
      f'а глобальный максимум на [0..15] при k={glob_max_16} (P={p_m16[glob_max_16]:.4f}) — второй период вращения!')

# Проверка M = 32 (половина базы) и M = 48 (3/4 базы) при n = 6
print('\nПроверка больших долей решений при n = 6 (N = 64):')
for M_big in [32, 48]:
    marked_big = list(rng6.choice(all_states_6, size=M_big, replace=False))
    probs_big = []
    for k in range(6):
        sv = Statevector.from_instruction(grover_circuit(n6, marked_big, k))
        p_dict = sv.probabilities_dict()
        probs_big.append(sum(p_dict.get(st, 0.0) for st in marked_big))
    best_k = int(np.argmax(probs_big))
    print(f'M = {M_big} (M/N = {M_big/N6:.2f}): P(k=0..5) = {[round(x, 4) for x in probs_big]} -> оптимально k = {best_k}')

### Анализ результатов Задания 6
* **Сравнение с теорией:** Для $M = 1, 2, 4, 8, 16$ первый локальный максимум $k_{\text{первый макс}}$ ($6, 4, 3, 2, 1$) точно совпадает с теоретической оценкой $k_{\text{теор}} = \text{round}\left(\frac{\pi}{4}\sqrt{\frac{N}{M}}\right)$.
* **Почему вероятность снова растёт после оптимума (пример $M=16$):** Геометрически каждая итерация Гровера поворачивает вектор состояния в двумерной плоскости на угол $2\theta$, где $\sin\theta = \sqrt{M/N}$. Вероятность успеха изменяется периодически как $P(k) = \sin^2((2k+1)\theta)$. После достижения первого пика ($P \approx 1$) при $(2k+1)\theta \approx \pi/2$ дальнейшие итерации «перекручивают» вектор мимо целевого подпространства (вероятность падает почти до $0$ при $(2k+1)\theta \approx \pi$), а затем продолжают вращение и достигают следующего максимума при $(2k+1)\theta \approx 3\pi/2$ (для $M=16$, $\theta = \pi/6$, пики при $k=1, 4, 7, 10, 13$, причём $P(4) = 1.0 > P(1) = 0.945$).
* **Правило выбора оптимального $k$:** Всегда нужно выбирать **первый локальный максимум** $k_{\text{opt}} \approx \frac{\pi}{4}\sqrt{\frac{N}{M}}$, так как он даёт максимальную (или близкую к $1$) вероятность за минимальное число обращений к оракулу и минимальную глубину схемы.
* **Случаи $M = 32$ ($M/N = 1/2$) и $M = 48$ ($M/N = 3/4$):**
  * При $M = 32$ ($\theta = \pi/4$) после $k=0$ вероятность равна $0.5$, а после $k=1$ угол равен $3\pi/4$, и вероятность снова $\sin^2(3\pi/4) = 0.5$ (и далее всегда $0.5$).
  * При $M = 48$ ($\theta = \pi/3$) при $k=0$ вероятность без всяких итераций равна $48/64 = 0.75$, а при $k=1$ падает до $\sin^2(\pi) = 0$! Таким образом, при $M \ge N/2$ **оптимально $k = 0$** (классический случайный выбор за 1 попытку эффективнее квантового поиска; алгоритм Гровера полезен только при $M \ll N$).

## Задание 7: Сравнение вариантов 1–10 ($n=5, N=32$) и влияние шума

In [ ]:
# Группа вариантов 1-10 (n = 5, N = 32), включая Вариант 8 (|01101>)
variants_n5 = {
    1: '01010',
    2: '10101',
    3: '11001',
    4: '00110',
    5: '11100',
    6: '00011',
    7: '10010',
    8: '01101',
    9: '10110',
    10: '01001',
}

n7 = 5
N7 = 2**n7
ks7 = list(range(16))
k_th_7 = round((np.pi / 4.0) * np.sqrt(N7))

print('Вариант | Состояние |  N | k_теор | k_opt_точн | k_opt_эксп (1024 shots) | P_точн(k_opt)')
print('-' * 82)

plt.figure(figsize=(9, 5))
k_exp_list = []

for var_id, state_w in variants_n5.items():
    p_ex_arr = []
    p_exp_arr = []
    for k in ks7:
        qc_k = grover_circuit(n7, [state_w], k)
        sv = Statevector.from_instruction(qc_k)
        p_ex_arr.append(sv.probabilities_dict().get(state_w, 0.0))

        qc_m = qc_k.copy()
        qc_m.measure_all()
        cnt = sim.run(qc_m, shots=1024, seed_simulator=500 * var_id + k).result().get_counts()
        p_exp_arr.append(cnt.get(state_w, 0) / 1024.0)

    k_opt_ex = int(np.argmax(p_ex_arr))
    k_opt_ep = int(np.argmax(p_exp_arr))
    k_exp_list.append(k_opt_ep)
    p_max_ex = p_ex_arr[k_opt_ex]

    print(f'   {var_id:2d}   |  |{state_w}>  | {N7} |   {k_th_7:2d}   |     {k_opt_ex:2d}     |            {k_opt_ep:2d}           |   {p_max_ex:.6f}')

    if var_id == 8:
        plt.plot(ks7, p_exp_arr, 'r-o', linewidth=2.5, label=f'Вариант 8 |{state_w}> (эксп)')
        plt.plot(ks7, p_ex_arr, 'k--', linewidth=2, label='Точная теория (Statevector)')
    else:
        plt.plot(ks7, p_exp_arr, alpha=0.45, linestyle='-', marker='.', label=f'Вар. {var_id} |{state_w}>')

plt.xticks(ks7)
plt.title('Кривые вероятности для группы вариантов 1–10 (n = 5, N = 32)')
plt.xlabel('Число итераций k')
plt.ylabel('Вероятность P(w)')
plt.legend(ncol=2, fontsize=8)
plt.grid(True)
plt.show()

print(f'\nРазброс k_opt_эксп по группе вариантов 1–10: min={min(k_exp_list)}, max={max(k_exp_list)}, значения={k_exp_list}')

In [ ]:
# 7.4 Моделирование двухкубитного деполяризующего шума на CX для Варианта 8 (n=5, |01101>) при p2 = 0.005 и 0.02
w_var8 = variants_n5[8]
p2_noise_levels = [0.0, 0.005, 0.02]

plt.figure(figsize=(8, 4.5))
for p2_val in p2_noise_levels:
    noise_m = NoiseModel()
    if p2_val > 0:
        noise_m.add_all_qubit_quantum_error(depolarizing_error(p2_val, 2), ['cx'])

    probs_noisy = []
    for k in ks7:
        qc_k = grover_circuit(n7, [w_var8], k)
        qc_k.measure_all()
        qc_t = transpile(qc_k, sim, basis_gates=['rz', 'sx', 'x', 'cx'], optimization_level=1)
        if k == 1:
            cx_per_iter = qc_t.count_ops().get('cx', 0)
            depth_per_iter = qc_t.depth()
        cnt = sim.run(qc_t, noise_model=noise_m if p2_val > 0 else None, shots=1024, seed_simulator=777 + k).result().get_counts()
        probs_noisy.append(cnt.get(w_var8, 0) / 1024.0)

    k_best_n = int(np.argmax(probs_noisy))
    print(f'Вариант 8 (|{w_var8}>), p2 = {p2_val:5.3f}: max P = {probs_noisy[k_best_n]:.4f} при k_opt = {k_best_n}')
    plt.plot(ks7, probs_noisy, marker='o', label=f'p2 = {p2_val}')

print(f'\nПри n=5 одна итерация Гровера после transpile содержит {cx_per_iter} вентилей CX и имеет глубину {depth_per_iter}.')
plt.xticks(ks7)
plt.title(f'Влияние двухкубитного шума p2 на алгоритм Гровера (Вариант 8, |{w_var8}>, n=5)')
plt.xlabel('Число итераций k')
plt.ylabel('Экспериментальная вероятность P(w)')
plt.legend()
plt.grid(True)
plt.show()

### Выводы к Заданию 7
1. **Зависит ли оптимальное число итераций от отмеченного состояния при $M=1$?**
   **Нет, не зависит.** Для всех состояний группы с одинаковым числом кубитов $n=5$ ($N=32$) начальная амплитуда каждого базисного состояния одинакова ($1/\sqrt{N}$), угол поворота $\theta = \arcsin(1/\sqrt{32})$ один и тот же, поэтому точная вероятность $P_{\text{точн}}(k)$ и точный оптимум $k_{\text{точн}} = 4$ ($P_{\text{точн}}(4) = 0.99918$) абсолютно идентичны для всех 10 вариантов. Небольшие отклонения $P_{\text{эксп}}$ от $P_{\text{точн}}$ объясняются исключительно конечным числом измерений (статистическим шумом выборки при $1024$ shots).
2. **Как шум меняет максимум $P$ и положение оптимума, и почему важна глубина схемы?**
   * При разложении через `transpile` многоуправляемый вентиль MCZ для $n=5$ превращается в цепочку из десятков двухкубитных вентилей CX (более 70 CX на одну итерацию Гровера).
   * С ростом уровня шума $p_2$ деполяризация быстро разрушает когерентность, из-за чего **максимум вероятности $P$ резко падает** (при $p_2 = 0.005$ пик снижается примерно до $0.45\text{--}0.55$, а при $p_2 = 0.02$ — до $\approx 0.10\text{--}0.15$), а осцилляции затухают, приближая распределение к равномерному ($1/32 \approx 0.031$).
   * **Положение оптимума $k_{\text{opt}}$ сдвигается влево** (к меньшим $k$, например с $k=4$ к $k=2\text{--}3$ или $k=1\text{--}2$), так как на поздних итерациях накопленный шум от глубокой схемы начинает разрушать состояние быстрее, чем поворот Гровера усиливает амплитуду. Именно поэтому глубина схемы критически важна: на реальных NISQ-устройствах алгоритм необходимо останавливать раньше теоретического $k_{\text{теор}}$, пока ошибка не превысила выигрыш от усиления амплитуды.